# 📅 2026-09-04 개발 노트 : 백필 첫 가동 — 라벨·활성화 누락 수정, 모집단 전환 대응(노출 게이트), 학생 모델 감사 설계

## 🎯 오늘의 목표 — "3월 3주차 이후 밀린 신작 전량 수집을 실제로 돌린다"

- [x] `weekly_pipeline --from 2026-03-16 --limit 500 --loop` 가동 (detached, 로그 파일 감시)
- [x] 1회차 적재 결과 검토 → **라벨 누락 + 활성화 누락** 발견·수정 (2회차부터 반영 확인)
- [x] "왜 이렇게 많이 나오냐" → 기존 4,190 = 인기순 샘플, 백필 = 전수 → **모집단 전환** 인지
- [x] 노출 정책 모듈 + Steam 리뷰 수 갱신 + 30일 재평가 (데이터 보존, 노출만 제어)
- [x] "5.4-mini 품질 괜찮다"는 판단의 근거 재점검 → 교사 vs 학생 **홀드아웃 감사 도구** 작성
- [x] 백필 후처리 원커맨드(`post_backfill`) + 한글 상세 커밋 3건
- [ ] 루프 종료 후 후처리 실행 → 홀드아웃 결과·리뷰 분포 보고 게이트 기준 확정 (다음)

> 오늘 배운 한 줄: **로그가 깨끗해도 끝단(서비스 노출)까지 도달했는지 봐야 하고, "분포가 비슷하다"는 같은 모집단일 때만 의미가 있다.**


## 1. 백필 가동 — 운영 관찰 기록

`docker compose exec -d batch sh -c "python -m embeddings.weekly_pipeline --from 2026-03-16 --limit 500 --loop --wait-timeout 60 > /app/data/backfill.log 2>&1"`

- `-d`라 프롬프트가 바로 돌아와 "끝났나?" 싶었지만 정상. 감시는 `tail -n 30 /app/data/backfill.log`.
- 크롤 1회차: 후보 13,095개까지 훑고 3월 16일 구간 도달 → 조기 종료 동작 확인. 신작 500 + 기존 pending 500 = 1,000건 배치.
- **1회차 배치 낙오**: 987~998/1000에서 정지 → 60분 타임아웃에 취소 요청 → OpenAI `cancelling`이 27분(문서상 최대 10분) → 999건 부분 수거. 이후 회차는 낙오 없이 3~46분 완주.
- 회차 흐름: 크롤 24분(스토어 페이징 15분 + 상세 1.5s/건) + 배치 3~46분 + 적재/임베딩 1분 ≈ 30~70분.
- 9회차까지 4,500건 적재, 실패 0. 10회차에 7월 9일 출시작 통과 → 3월까지 9~10회차 더.

| 회차 | 신작+pending | 배치 | gem 평균 | 활성화 |
|---|---|---|---|---|
| 1 | 500+500 | 999/1000 (타임아웃 취소) | 43.1 | 0 (라벨 버그) |
| 2 | 500+151 | 651/651 | 44.2 | 641 |
| 3 | 500+0 | 500/500 | 42.9 | 498 |
| 4~9 | 500+0 | 500/500 | 43.4~45.1 | 494~498 |


## ⭐ 2. 1회차 적재는 성공인데 임베딩 "대상 없음" — 두 겹의 누락

**증상:** `load#1` UPSERT 999/999 성공, 49지표 완전체 100%. 그런데 `embed#1`이 `embedding NULL인 대상이 없습니다`로 즉시 종료.

**원인 1 — 라벨:** `batch_processor --version` 기본값이 교사 라벨 `gpt5.4-batch-v1`. 파이프라인이 옵션을 안 넘겨 5.4-mini 결과 999건이 교사 데이터로 기록됨.
`generate_embeddings`는 `analysis_method='fewshot_5.4based'`만 대상으로 잡아 전부 건너뜀.

**원인 2 — 활성화:** 임베딩 생긴 뒤 `is_active=TRUE`로 켜는 단계가 **어디에도 없음**. `batch_processor`는 적재 시점에 embedding이 없어 항상 비활성으로 남기고, 그 뒤를 받는 코드가 없었다.
→ 라벨을 고쳐도 몇 천 개를 받든 서비스엔 0개 노출되는 구조.

**수정 (루프 재시작 없이 2회차부터 반영 — 각 단계가 서브프로세스라 디스크의 새 코드를 읽음):**
- `batch_processor`: `--version` 생략 시 결과 JSONL의 **응답 모델명으로 라벨 판정** (gpt-5.4 본모델만 교사, mini/nano는 학생). 플래그 실수가 데이터 오염으로 이어질 수 없게 provenance를 데이터에서 읽는다.
- `generate_embeddings`: 임베딩 후 활성화 단계 추가(멱등, 학생 라벨만, 소프트웨어 49건 무접촉). 검증 출력 수백 줄 → 최근 10건.
- `relabel_version.py`(신규): 1회차 999건 복구용. JSONL의 custom_id → app_id → games/game_metrics 라벨만 교체.
- 오케스트레이터(`weekly_pipeline`)는 메모리에 떠 있어 이번 루프엔 반영 안 됨 → `--version` 명시는 다음 실행부터.

**검증:** 2회차 로그에 `응답 모델: gpt-5.4-mini-2026-03-17=651 → extraction_version: fewshot_5.4based (응답 모델명으로 판정)`, `대상 (신작만): 651건`, `활성화: 641건`. 10건은 설명 40자 미만으로 원본 필터 규칙대로 스킵.


## ⭐ 3. "4,190개가 3월까지 전체인데 왜 이렇게 많이 나와?" — 모집단 전환

크롤러 로그의 Steam 전체 게임 수 **120,254개**. 기존 수집기(`history/collect_gems.py`)는 목표 5,000개를 태그별·인기순(topsellers)·2018~2024 연도별 검색으로 채운 **인기작 편향 샘플(3.5%)**.
백필은 3월 16일 이후 출시작을 조건 없이 **전수** 수집 (연 1만 8천 개 출시 → 5.5개월 ≈ 8천 개, 1회차 후보 13,095개와 부합).

**따라오는 문제 두 가지:**
1. 크롤러 필터는 type=game + 설명 11자, 둘뿐 → 리뷰 0개 무명작·에셋 플립이 그대로 들어와 추천 잡음 위험.
2. 크롤러가 쓰는 appdetails엔 리뷰 집계가 없어 신작 **전원 `review_count=0`, `steam_positive_ratio=NULL`**. 리뷰로 거르고 싶어도 데이터가 없었다.

**결정 (취지 = 게임의 가능성 보존):** 데이터는 지우지 않는다. 분석·임베딩은 유지하고 **노출(is_active)만 리뷰 수로 제어**, 리뷰가 붙으면 다시 켠다. 기존 4,190개도 수집 당시 인기/리뷰 기준을 거쳤으니 일관됨.

**구현:**
- `exposure_policy.py`: 노출 규칙 단일 모듈. metrics + embedding + `review_count >= MIN_REVIEWS_FOR_EXPOSURE`(기본 10 = Steam이 리뷰 점수를 표시하는 최소 개수, `.env` 조정). 학생 라벨만 대상.
- `refresh_reviews.py`: Steam appreviews 요약 API로 리뷰 수/긍정비율 채움 + 게이트. 조회 이력은 `review_refresh_log` 테이블(games 스키마 무접촉). `--new` / `--recheck --stale-days 30`(비활성 재평가 → 재활성화) / `--gate-only`. 1초/건, 8천 개 ≈ 2.5h.
- `generate_embeddings` 활성화가 같은 규칙 사용 → 이후 회차 embed에서 활성화 0건이 **정상** (리뷰 미조회). `weekly_pipeline`에 `reviews#i` 단계 + 루프 끝 `recheck` 연결(다음 실행부터).


## ⭐ 4. "5.4-mini 품질 괜찮다"는 판단, 근거가 약했다 — 홀드아웃 감사

내가 "캘리브레이션 유지"라고 한 근거는 회차별 gem 평균이 43~45로 일정하다는 것 하나. 그건 **드리프트 감시**일 뿐 품질 검증이 아니다.
- 교사 데이터 gem 평균 75.7 vs 학생 44 — 모집단이 다르니(선별 vs 전수) 분포가 달라야 정상이고, 그렇다면 "평균이 비슷하다"는 안심 근거가 아니라 **few-shot 12개의 사전분포로 답이 끌려가는 신호**일 수도 있다.
- 어제(0903) 실험도 동일 10게임에서 4o-mini vs 5.4-mini의 **변별력 비교**였고, 교사 정답과의 직접 비교는 아니었다.
- 입력이 이름·장르·설명 블라인드라 설명만 그럴싸한 에셋 플립도 높은 gem을 받을 수 있는 구조 → 이건 모델이 아니라 리뷰 게이트가 보완.

**진짜 검증 = 같은 게임을 교사와 학생이 각각 매긴 값 직접 비교.** `embeddings/audit_student.py`:
- `--make-holdout 150`: 교사 게임을 gem 5분위 층화 샘플(few-shot 예시 제외). 정답은 블라인드 CSV와 분리 저장.
- (학생 모델로 블라인드 분석, DB 미접촉, $0.25)
- `--compare`: 49 수치 MAE·상관, 9 불리언 일치율, gem MAE·스피어만·low/mid/high 혼동표, confidence σ(0.05 미만 = 평탄화). 휴리스틱 판정: gem MAE ≤ 8 & ρ ≥ 0.7 & 구간 일치 ≥ 70% & MAE>2.5 지표 ≤ 5개.
  비교 후 결과 JSONL을 `data/audit/`로 이동 — `batch_processor`로 잘못 적재하면 **교사 값이 학생 값으로 덮이는** 사고 방지.
- `--new-sample`: 신작 gem 상위/하위/무작위 15건 눈검수 CSV.

**통과 못 하면:** few-shot 예시 구성 조정 → 그래도 안 되면 신작 일부를 교사 모델로 재분석까지 열어둠.


## 5. 후처리 원커맨드 + 운영 메모

**`embeddings/post_backfill.py`** — 루프 끝난 뒤 순서대로: 1회차 라벨 복구 → 임베딩 → 홀드아웃 감사(배치, 실패 시 sync 폴백) → 리뷰 갱신+게이트(2~3h) → 눈검수 샘플. `--wait`면 `/proc` 스캔으로 `weekly_pipeline` 종료를 기다림. 전 출력은 `data/audit/post_backfill_report.txt`.

```
docker compose exec -d batch sh -c "python -m embeddings.post_backfill --wait > /app/data/post_backfill.log 2>&1"
```

**운영 메모:**
- `--sync`/홀드아웃 배치를 루프 도는 중에 돌리면 `data/batch_output_*.jsonl`이 루프의 "최신 결과 파일" 탐지에 섞일 수 있다 → 반드시 루프 종료 후.
- Steam 리뷰 조회도 크롤러와 같은 IP로 두드리면 429 → 루프 종료 후.
- 데스크탑 폴더 공유 상태에서 노트북으로 이어 작업 가능(세션은 클라우드, 폴더 접근은 데스크탑 앱 경유). 도커 명령은 데스크탑에서.
- 오늘 커밋 3건(한글 상세): 라벨·활성화 수정 / 노출 게이트·리뷰·감사·후처리 / 백필 루프·타임아웃·단계 연결.

## 📋 다음 할 일
- ⬜ 루프 종료 확인 → `post_backfill --wait` 실행 → 보고서 확인
- ⬜ 홀드아웃 결과로 학생 모델 판정 (통과/예시 조정/교사 재분석)
- ⬜ 리뷰 수 분포 보고 `MIN_REVIEWS_FOR_EXPOSURE` 기준 확정 (기본 10)
- ⬜ `git push` (Vercel/Railway 자동 배포 — 백엔드 변경은 embeddings/만이라 서비스 영향 없음)
- ⬜ 주간 스케줄 등록 `setup_weekly_task.ps1`
- ⬜ Railway 배포 실패 알림, Umami 운영 연결 확인, 포트폴리오 ⑩⑪ [채우기]
